In [1]:
import json
import pandas as pd

valid_records = []
invalid_record_count = 0

with open("results/20260812_202653.records.jsonl", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
            
        try:
            valid_records.append(json.loads(line))
        except json.JSONDecodeError:
            invalid_record_count += 1
            # Skip the corrupted line (usually the very last one where you killed the script)
            pass

# pd.json_normalize automatically flattens nested dictionaries.
# A JSON structure like {"result": {"score": 0.8}} becomes a column named "result.score"
df = pd.json_normalize(valid_records)

print(f"Loaded {len(df)}/{len(df) + invalid_record_count} valid records.")

Loaded 181/181 valid records.


In [2]:
df.columns

Index(['setting.prompt.setting.system_prompt',
       'setting.prompt.setting.response_format.type',
       'setting.prompt.setting.prompt_format', 'setting.prompt.name',
       'setting.model', 'input.text', 'input.language.source_language',
       'input.language.target_language', 'result.translation', 'result.rating',
       'metadata.created_at', 'metadata.success', 'metadata.error',
       'metadata.elapsed_seconds'],
      dtype='str')

In [3]:
df['setting.model']

0      qwen3-8b
1      qwen3-8b
2      qwen3-8b
3      qwen3-8b
4      qwen3-8b
         ...   
176    qwen3-8b
177    qwen3-8b
178    qwen3-8b
179    qwen3-8b
180    qwen3-8b
Name: setting.model, Length: 181, dtype: str

In [5]:
pd.set_option('display.max_colwidth', 60)
df[["input.text", "result.translation", "result.rating", "setting.model", "setting.prompt.name"]].sort_values("result.rating")

,input.text,result.translation,result.rating,setting.model,setting.prompt.name
29,Those who practise magical arts of blowing on knots-the ...,Yang mempraktikkan seni magis menghembuskan napas ke sim...,-0.621145,qwen3-8b,verbose
27,Those who practise magical arts of blowing on knots-the ...,Orang-orang yang mempraktikkan ilmu sihir menghembuskan ...,-0.529700,qwen3-8b,minimal
28,Those who practise magical arts of blowing on knots-the ...,Yang mempraktikkan seni magis meniup simpul-simpul - sih...,-0.369753,qwen3-8b,professional
58,An angel got her strangled by the very rope she used to ...,Seorang malaikat menggantungkan tali yang sama yang digu...,-0.338347,qwen3-8b,professional
141,"'_Asfin Makool'_, i.e., straw eaten up, i.e., rendered l...","'_Asfin Makool'_ , yaitu, rumput yang telah dimakan, yai...",-0.329658,qwen3-8b,minimal
...,...,...,...,...,...
72,"See <{[""Q"", ""109:1""]}>","Lihat <{[""Q"", ""109:1""]}>",0.376887,qwen3-8b,minimal
78,"See <{[""Q"", ""109:1""]}>","Lihat <{[""Q"", ""109:1""]}>",0.376887,qwen3-8b,minimal
52,When Abu-Lahab will be in the hell-fire and then he woul...,"Ketika Abu-Lahab berada di api neraka, nanti dia akan me...",0.387317,qwen3-8b,professional
63,"See <{[""Q"", ""110:1""]}>","Lihat <{[""Q"", ""110:1""]}>",0.415419,qwen3-8b,minimal


In [8]:
display(df.groupby(['setting.prompt.name', 'setting.model'])['metadata.elapsed_seconds'].describe())
display(df.groupby(['setting.prompt.name', 'setting.model'])['result.rating'].describe())

,,count,mean,std,min,25%,50%,75%,max
setting.prompt.name,setting.model,,,,,,,,
minimal,qwen3-8b,61.0,6.644854,5.328131,2.798317,3.351026,4.327142,7.881365,31.159906
professional,qwen3-8b,60.0,6.586182,5.298610,2.815771,3.326645,4.284510,8.161167,31.186950
verbose,qwen3-8b,60.0,6.789303,5.433011,2.788934,3.368427,4.369940,7.904989,31.706455


,,count,mean,std,min,25%,50%,75%,max
setting.prompt.name,setting.model,,,,,,,,
minimal,qwen3-8b,61.0,0.071637,0.220833,-0.529700,-0.093439,0.052275,0.253908,0.415419
professional,qwen3-8b,60.0,0.055823,0.225369,-0.369753,-0.129028,0.053291,0.234320,0.415419
verbose,qwen3-8b,60.0,-0.001923,0.189815,-0.621145,-0.133392,0.012901,0.133496,0.374963


In [9]:
val = df['input.text'].values
val

[v for v in val if any(k in v for k in '<{[(')]

['Be the evil doers, the Jinns or human beings. It is a well-known fact that among the forces hidden from the human eye, which are active in the world, there are good and also bad ones among them. Jinns, like Satan are spiritual beings<{["F", 1]}>.\n\n**The End**',
 'Be the evil doers, the Jinns or human beings. It is a well-known fact that among the forces hidden from the human eye, which are active in the world, there are good and also bad ones among them. Jinns, like Satan are spiritual beings<{["F", 1]}>.\n\n**The End**',
 'Be the evil doers, the Jinns or human beings. It is a well-known fact that among the forces hidden from the human eye, which are active in the world, there are good and also bad ones among them. Jinns, like Satan are spiritual beings<{["F", 1]}>.\n\n**The End**',
 "'_Falaq'_ i.e., cleaving, here means cleaving of the darkness, i.e., dawn - it may mean destroyer of the evil effects of black art, i.e., the darkness or the calamitous effect of witchcraft. This Sura

In [11]:
def check_citations_intact(source: str, translation: str) -> bool:
    import re
    pattern = re.compile(r'<\{\["([\w]+)"\s*,\s*("[\w:]+"|\d+)\]\}>')
    return pattern.findall(source) == pattern.findall(translation)

In [12]:
print(check_citations_intact('See <{["Q", "112:3"]}>', 'Lihat <{["Q", "112:3"]}>'))
print(check_citations_intact('and is followed by the Successor,<{["F", 1]}>', 'dan diikuti Penerusnya,<{["F", 1]}>'))

True
True


In [15]:
intact_citations = df.apply(lambda row: check_citations_intact(row['input.text'], row['result.translation']), axis=1)
display(intact_citations.describe())
df[~intact_citations].groupby(['setting.prompt.name', 'setting.model'])[
    ['setting.prompt.name', 'setting.model', 'input.text', 'result.translation']
].describe()

count      181
unique       2
top       True
freq       180
dtype: object

setting.prompt.name                       \
                                                count unique      top freq   
setting.prompt.name setting.model                                            
minimal             qwen3-8b                        1      1  minimal    1   

                                  setting.model                        \
                                          count unique       top freq   
setting.prompt.name setting.model                                       
minimal             qwen3-8b                  1      1  qwen3-8b    1   

                                  input.text         \
                                       count unique   
setting.prompt.name setting.model                     
minimal             qwen3-8b               1      1   

                                                                                                \
                                                                                           top   
setting.prompt.name setting.model                                                                
minimal             qwen3-8b       When the Quraysh have been granted such great blessings ...   

                                       result.translation         \
                                  freq              count unique   
setting.prompt.name setting.model                                  
minimal             qwen3-8b         1                  1      1   

                                                                                                \
                                                                                           top   
setting.prompt.name setting.model                                                                
minimal             qwen3-8b       Ketika Quraysh telah diberikan berbagai berkat yang besa...   

                                        
                                  freq  
setting.prompt.name setting.model       
minimal             qwen3-8b         1

In [39]:
df[~intact_citations]['input.text'].apply(print)
print("====")
df[~intact_citations]['result.translation'].apply(print)

When the Quraysh have been granted such great blessings they must gratefully worship the Lord of the Ka'ba, who in spite of their land being only a desert and thus unproductive, provides them with every kind of good food and the other provisions of life through merchandise and the pilgrimage which brings to their very door, their needs through the great multitudes of the visitors who come there also to buy and sell, and God had protected them from the most formidable enemy Abraha, but for God's help they would have been destroyed and their city would have been lost, the Ka'ba, on account of which they are honoured and respected, their city would have been laid waste by the enemy, they with their wives and children would have been taken as captives, and enslaved and sold away to the others and thus they would have been tom away from their kith and kin and their tribe as a whole would have been disintegrated and wiped out of existence.

This was an open admonition to the Quraysh who had 

126    None
Name: result.translation, dtype: object

In [59]:
with pd.option_context('display.max_colwidth', None):
    display(df[["input.text", "result.translation", "result.rating", "setting.model", "setting.prompt.name"]
        ].sort_values("result.rating").tail(30))

,input.text,result.translation,result.rating,setting.model,setting.prompt.name
162,"See <{[""Q"", ""104:6""]}>","Lihat <{[""Q"", ""104:6""]}>",0.307317,qwen3-8b,minimal
169,"See <{[""Q"", ""104:8""]}>","Lihat <{[""Q"", ""104:8""]}>",0.309938,qwen3-8b,professional
168,"See <{[""Q"", ""104:8""]}>","Lihat <{[""Q"", ""104:8""]}>",0.309938,qwen3-8b,minimal
51,When Abu-Lahab will be in the hell-fire and then he would realise the return for his disbelief and tyranny and know of what value was his wealth.,"Ketika Abu-Lahab berada di api neraka, nanti dia akan menyadari balasan atas kekafiran dan siksa yang dia lakukan, dan dia akan mengetahui seberapa bernilainya harta benda yang dia miliki.",0.314910,qwen3-8b,minimal
147,"See <{[""Q"", ""104:1""]}>","Lihat <{[""Q"", ""104:1""]}>",0.335875,qwen3-8b,minimal
148,"See <{[""Q"", ""104:1""]}>","Lihat <{[""Q"", ""104:1""]}>",0.335875,qwen3-8b,professional
120,"The protection, safety and the honour, the Quraysh enjoyed was due to the Holy Ka'ba being in their city and their being its guardians.","Perlindungan, keselamatan, dan kehormatan yang dinikmati oleh Quraysh adalah karena Ka'ba yang suci berada di kota mereka dan mereka menjadi penjaganya.",0.339310,qwen3-8b,minimal
121,"The protection, safety and the honour, the Quraysh enjoyed was due to the Holy Ka'ba being in their city and their being its guardians.","Perlindungan, keselamatan, dan kehormatan yang dinikmati oleh Quraysh adalah karena Ka'ba yang suci berada di kota mereka dan mereka menjadi penjaganya.",0.339310,qwen3-8b,professional
122,"The protection, safety and the honour, the Quraysh enjoyed was due to the Holy Ka'ba being in their city and their being its guardians.","Perlindungan, keselamatan, dan kehormatan yang dinikmati oleh Quraysh adalah karena Ka'ba yang suci berada di kota mereka dan mereka menjadi penjaganya.",0.339310,qwen3-8b,verbose
135,"God sent upon the army, a huge host of some tiny birds with small stones in their beaks.",Tuhan mengirimkan kepada pasukan itu sebuah rombongan besar dari beberapa burung kecil yang memiliki batu kecil di paruh mereka.,0.343960,qwen3-8b,minimal


,input.text,result.translation,result.rating,setting.model,setting.prompt.name
29,Those who practise magical arts of blowing on knots-the favourite witchcraft generally practised by perverted woman which creates psychological disturbances.,Yang mempraktikkan seni magis menghembuskan napas ke simpul-simpul-kesenian penyihir yang paling disukai umumnya dilakukan oleh wanita yang tergoda yang menciptakan gangguan psikologis.,-0.621145,qwen3-8b,verbose
27,Those who practise magical arts of blowing on knots-the favourite witchcraft generally practised by perverted woman which creates psychological disturbances.,"Orang-orang yang mempraktikkan ilmu sihir menghembuskan napas ke simpul-simpul—sihir penyihir yang paling disukai umumnya dilakukan oleh wanita yang tergoda, yang menciptakan gangguan psikologis.",-0.529700,qwen3-8b,minimal
28,Those who practise magical arts of blowing on knots-the favourite witchcraft generally practised by perverted woman which creates psychological disturbances.,Yang mempraktikkan seni magis meniup simpul-simpul - sihir penyihir yang paling disukai umumnya dilakukan oleh wanita yang terdistorsi yang menciptakan gangguan psikologis.,-0.369753,qwen3-8b,professional
58,"An angel got her strangled by the very rope she used to hang around her neck, and she died. On the Day of Judgment, the wicked woman will be with a rope of the hell-fire hanging around her neck.","Seorang malaikat menggantungkan tali yang sama yang digunakannya menggantungkan lehernya sendiri, dan dia mati. Pada Hari Kiamat, wanita jahat itu akan berada di sisi tali api neraka yang menggantungkan lehernya.",-0.338347,qwen3-8b,professional
141,"'_Asfin Makool'_, i.e., straw eaten up, i.e., rendered lifeless, useless and loathsome - like the refuse of an eaten food.","'_Asfin Makool'_ , yaitu, rumput yang telah dimakan, yaitu, diubah menjadi tidak bernyawa, tidak berguna, dan menjijikkan - seperti sisa makanan yang telah dimakan.",-0.329658,qwen3-8b,minimal
20,"'_Falaq'_ i.e., cleaving, here means cleaving of the darkness, i.e., dawn - it may mean destroyer of the evil effects of black art, i.e., the darkness or the calamitous effect of witchcraft. This Surah was revealed to undo any evil effects of any witchcraft or sorcery.\n\nThe Tradition, the Holy Prophet being enchanted by the sorcerers in such a manner that he was himself unconscious what he was actually doing - reduces the Last Prophet of God to such a state that the Rod of Moses would be superior to him to undo the sorcery.\n\nThese sorts of Traditions undoubtedly have been fabricated by the mischievous elements who wanted to reduce the sublime position of the Holy Prophet to their own level of character and belief. Of course, there are traditions about the utility of the recitation of this and the previous, and the next suras in dispelling the possible effect of any witchcraft. (A.P.).","'_Falaq'_ yaitu, memisahkan, di sini berarti memisahkan gelap, yaitu, fajar - mungkin berarti penghancur dari efek jahat sihir hitam, yaitu, gelap atau efek musibah sihir. Surah ini diturunkan untuk menghilangkan efek jahat dari sihir atau perdukunan apa pun.\n\nAdat-istiadat, Nabi yang suci terkena sihir oleh para dukun sedemikian rupa hingga ia sendiri tidak menyadari apa yang sedang ia lakukan - mengurangi Nabi terakhir Tuhan sampai pada kondisi di mana tongkat Musa akan lebih unggul darinya untuk menghilangkan sihir tersebut.\n\nAdat-istiadat semacam ini jelas telah dibuat oleh elemen-elemen yang suka berbuat jahat yang ingin menurunkan posisi mulia Nabi yang suci sampai ke tingkat karakter dan keyakinan mereka sendiri. Jelas saja, ada adat-istiadat tentang manfaat membaca Surah ini dan Surah-surah sebelumnya serta sesudahnya dalam menghilangkan efek sihir apa pun yang mungkin terjadi. (A.P.).",-0.324726,qwen3-8b,verbose
19,"'_Falaq'_ i.e., cleaving, here means cleaving of the darkness, i.e., dawn - it may mean destroyer of the evil effects of black art, i.e., the darkness or the calamitous effec

In [60]:
with pd.option_context('display.max_colwidth', None):
    display(df[["input.text", "result.translation", "result.rating", "setting.model", "setting.prompt.name"]
        ].sort_values("result.rating").head(30))

,input.text,result.translation,result.rating,setting.model,setting.prompt.name
29,Those who practise magical arts of blowing on knots-the favourite witchcraft generally practised by perverted woman which creates psychological disturbances.,Yang mempraktikkan seni magis menghembuskan napas ke simpul-simpul-kesenian penyihir yang paling disukai umumnya dilakukan oleh wanita yang tergoda yang menciptakan gangguan psikologis.,-0.621145,qwen3-8b,verbose
27,Those who practise magical arts of blowing on knots-the favourite witchcraft generally practised by perverted woman which creates psychological disturbances.,"Orang-orang yang mempraktikkan ilmu sihir menghembuskan napas ke simpul-simpul—sihir penyihir yang paling disukai umumnya dilakukan oleh wanita yang tergoda, yang menciptakan gangguan psikologis.",-0.529700,qwen3-8b,minimal
28,Those who practise magical arts of blowing on knots-the favourite witchcraft generally practised by perverted woman which creates psychological disturbances.,Yang mempraktikkan seni magis meniup simpul-simpul - sihir penyihir yang paling disukai umumnya dilakukan oleh wanita yang terdistorsi yang menciptakan gangguan psikologis.,-0.369753,qwen3-8b,professional
58,"An angel got her strangled by the very rope she used to hang around her neck, and she died. On the Day of Judgment, the wicked woman will be with a rope of the hell-fire hanging around her neck.","Seorang malaikat menggantungkan tali yang sama yang digunakannya menggantungkan lehernya sendiri, dan dia mati. Pada Hari Kiamat, wanita jahat itu akan berada di sisi tali api neraka yang menggantungkan lehernya.",-0.338347,qwen3-8b,professional
141,"'_Asfin Makool'_, i.e., straw eaten up, i.e., rendered lifeless, useless and loathsome - like the refuse of an eaten food.","'_Asfin Makool'_ , yaitu, rumput yang telah dimakan, yaitu, diubah menjadi tidak bernyawa, tidak berguna, dan menjijikkan - seperti sisa makanan yang telah dimakan.",-0.329658,qwen3-8b,minimal
20,"'_Falaq'_ i.e., cleaving, here means cleaving of the darkness, i.e., dawn - it may mean destroyer of the evil effects of black art, i.e., the darkness or the calamitous effect of witchcraft. This Surah was revealed to undo any evil effects of any witchcraft or sorcery.\n\nThe Tradition, the Holy Prophet being enchanted by the sorcerers in such a manner that he was himself unconscious what he was actually doing - reduces the Last Prophet of God to such a state that the Rod of Moses would be superior to him to undo the sorcery.\n\nThese sorts of Traditions undoubtedly have been fabricated by the mischievous elements who wanted to reduce the sublime position of the Holy Prophet to their own level of character and belief. Of course, there are traditions about the utility of the recitation of this and the previous, and the next suras in dispelling the possible effect of any witchcraft. (A.P.).","'_Falaq'_ yaitu, memisahkan, di sini berarti memisahkan gelap, yaitu, fajar - mungkin berarti penghancur dari efek jahat sihir hitam, yaitu, gelap atau efek musibah sihir. Surah ini diturunkan untuk menghilangkan efek jahat dari sihir atau perdukunan apa pun.\n\nAdat-istiadat, Nabi yang suci terkena sihir oleh para dukun sedemikian rupa hingga ia sendiri tidak menyadari apa yang sedang ia lakukan - mengurangi Nabi terakhir Tuhan sampai pada kondisi di mana tongkat Musa akan lebih unggul darinya untuk menghilangkan sihir tersebut.\n\nAdat-istiadat semacam ini jelas telah dibuat oleh elemen-elemen yang suka berbuat jahat yang ingin menurunkan posisi mulia Nabi yang suci sampai ke tingkat karakter dan keyakinan mereka sendiri. Jelas saja, ada adat-istiadat tentang manfaat membaca Surah ini dan Surah-surah sebelumnya serta sesudahnya dalam menghilangkan efek sihir apa pun yang mungkin terjadi. (A.P.).",-0.324726,qwen3-8b,verbose
19,"'_Falaq'_ i.e., cleaving, here means cleaving of the darkness, i.e., dawn - it may mean destroyer of the evil effects of black art, i.e., the darkness or the calamitous effec